**Load the configuration**

In [0]:
from pathlib import Path

import yaml
from pyspark.sql import functions as F


config_path = Path.cwd().parent / "config" / "cohort.yml"

with config_path.open("r") as file:
    config = yaml.safe_load(file)

cohort_config = config["cohort"]

cohort_id = cohort_config["id"]
expected_count = cohort_config["expected_patient_count"]
patient_ids = cohort_config["patient_ids"]
inclusion_rule = cohort_config["inclusion_rule"]

print(f"Cohort: {cohort_id}")
print(f"Configured patients: {len(patient_ids)}")

In [0]:
print(Path.cwd())

**Check duplicates and count**

In [0]:
if len(patient_ids) != expected_count:
    raise ValueError(
        f"Expected {expected_count} patients, "
        f"but configuration contains {len(patient_ids)}"
    )

if len(patient_ids) != len(set(patient_ids)):
    raise ValueError("cohort.yml contains duplicate patient IDs")

print("Configuration count and uniqueness: PASS")

**Confirm all configured patients exist**

In [0]:
bronze_patients = spark.table("patient_kg_dev.bronze.patients")

configured_ids_df = spark.createDataFrame(
    [(patient_id,) for patient_id in patient_ids],
    ["patient_id"],
)

existing_ids_df = bronze_patients.select(
    F.col("Id").alias("patient_id")
)

missing_ids_df = configured_ids_df.join(
    existing_ids_df,
    on="patient_id",
    how="left_anti",
)

missing_ids = [
    row["patient_id"]
    for row in missing_ids_df.collect()
]

if missing_ids:
    raise ValueError(f"Configured patients not found: {missing_ids}")

print("All configured patient IDs exist: PASS")

**Confirm the fixed IDs still match the rule**

In [0]:
bronze_conditions = spark.table(
    "patient_kg_dev.bronze.conditions"
)

rule_ids_df = (
    bronze_conditions
    .filter(
        (F.col("SYSTEM") == inclusion_rule["code_system"])
        & (F.col("CODE") == inclusion_rule["code"])
    )
    .select(F.col("PATIENT").alias("patient_id"))
    .distinct()
)

configured_not_rule_df = configured_ids_df.join(
    rule_ids_df,
    on="patient_id",
    how="left_anti",
)

rule_not_configured_df = rule_ids_df.join(
    configured_ids_df,
    on="patient_id",
    how="left_anti",
)

configured_not_rule = [
    row["patient_id"]
    for row in configured_not_rule_df.collect()
]

rule_not_configured = [
    row["patient_id"]
    for row in rule_not_configured_df.collect()
]

if configured_not_rule or rule_not_configured:
    raise ValueError(
        "Configured cohort does not match selection rule. "
        f"Configured but not rule-derived: {configured_not_rule}. "
        f"Rule-derived but not configured: {rule_not_configured}."
    )

print("Configured IDs match the coded inclusion rule: PASS")

**Create the cohort membership table**

In [0]:
cohort_membership_df = (
    configured_ids_df
    .withColumn("cohort_id", F.lit(cohort_id))
    .withColumn(
        "inclusion_dataset",
        F.lit(inclusion_rule["dataset"]),
    )
    .withColumn(
        "inclusion_code_system",
        F.lit(inclusion_rule["code_system"]),
    )
    .withColumn(
        "inclusion_code",
        F.lit(inclusion_rule["code"]),
    )
    .withColumn(
        "cohort_version",
        F.lit(cohort_config["version"]),
    )
    .select(
        "cohort_id",
        "cohort_version",
        "patient_id",
        "inclusion_dataset",
        "inclusion_code_system",
        "inclusion_code",
    )
)

(
    cohort_membership_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "patient_kg_dev.staged.cohort_membership"
    )
)

**Verifying Cohort**

In [0]:
display(
    spark.table(
        "patient_kg_dev.staged.cohort_membership"
    ).orderBy("patient_id")
)

**Dataset definitions**

In [0]:
DATASETS = {
    "patients": {
        "source_table": "patient_kg_dev.bronze.patients",
        "patient_column": "Id",
    },
    "encounters": {
        "source_table": "patient_kg_dev.bronze.encounters",
        "patient_column": "PATIENT",
    },
    "conditions": {
        "source_table": "patient_kg_dev.bronze.conditions",
        "patient_column": "PATIENT",
    },
    "medications": {
        "source_table": "patient_kg_dev.bronze.medications",
        "patient_column": "PATIENT",
    },
    "procedures": {
        "source_table": "patient_kg_dev.bronze.procedures",
        "patient_column": "PATIENT",
    },
    "observations": {
        "source_table": "patient_kg_dev.bronze.observations",
        "patient_column": "PATIENT",
    },
    "careplans": {
        "source_table": "patient_kg_dev.bronze.careplans",
        "patient_column": "PATIENT",
    },
}

**Staging function:**

In [0]:
def stage_dataset(
    dataset_name: str,
    source_table: str,
    patient_column: str,
):
    source_df = spark.table(source_table)

    staged_df = (
        source_df.alias("source")
        .join(
            F.broadcast(configured_ids_df).alias("cohort"),
            F.col(f"source.{patient_column}")
            == F.col("cohort.patient_id"),
            "inner",
        )
        .select("source.*")
    )

    expected_source_count = (
        source_df
        .join(
            F.broadcast(configured_ids_df),
            F.col(patient_column)
            == F.col("patient_id"),
            "inner",
        )
        .count()
    )

    target_table = (
        f"patient_kg_dev.staged.{dataset_name}"
    )

    (
        staged_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    target_count = spark.table(target_table).count()

    if expected_source_count != target_count:
        raise RuntimeError(
            f"Staging count mismatch for {dataset_name}: "
            f"expected={expected_source_count}, "
            f"actual={target_count}"
        )

    return {
        "dataset": dataset_name,
        "expected_count": expected_source_count,
        "staged_count": target_count,
        "status": "PASS",
    }

**Run-Staging**

In [0]:
staging_results = []

for dataset_name, dataset_config in DATASETS.items():
    print(f"Staging {dataset_name}...")

    result = stage_dataset(
        dataset_name=dataset_name,
        source_table=dataset_config["source_table"],
        patient_column=dataset_config["patient_column"],
    )

    staging_results.append(result)

display(
    spark.createDataFrame(staging_results)
    .orderBy("dataset")
)

In [0]:
%sql
SHOW TABLES IN patient_kg_dev.staged;

**Validate important edge cases**

1. Encounterless observations

In [0]:
%sql
SELECT COUNT(*) AS encounterless_observations
FROM patient_kg_dev.staged.observations
WHERE ENCOUNTER IS NULL
   OR TRIM(ENCOUNTER) = '';

2. Duplicate observation content

In [0]:
%sql
SELECT
    SUM(occurrence_count - 1) AS duplicate_rows_beyond_first
FROM (
    SELECT
        _row_content_sha256,
        COUNT(*) AS occurrence_count
    FROM patient_kg_dev.staged.observations
    GROUP BY _row_content_sha256
    HAVING COUNT(*) > 1
);

3. Patient membership

In [0]:
%sql
SELECT COUNT(DISTINCT Id)
FROM patient_kg_dev.staged.patients;

In [0]:
%sql
SELECT 'patients' AS dataset, COUNT(*) AS rows
FROM patient_kg_dev.staged.patients
UNION ALL
SELECT 'encounters', COUNT(*) FROM patient_kg_dev.staged.encounters
UNION ALL
SELECT 'conditions', COUNT(*) FROM patient_kg_dev.staged.conditions
UNION ALL
SELECT 'medications', COUNT(*) FROM patient_kg_dev.staged.medications
UNION ALL
SELECT 'procedures', COUNT(*) FROM patient_kg_dev.staged.procedures
UNION ALL
SELECT 'observations', COUNT(*) FROM patient_kg_dev.staged.observations
UNION ALL
SELECT 'careplans', COUNT(*) FROM patient_kg_dev.staged.careplans;

In [0]:
%sql
SELECT COUNT(*) AS encounterless
FROM patient_kg_dev.staged.observations
WHERE ENCOUNTER IS NULL OR TRIM(ENCOUNTER) = '';